Expected format

```py
{
    "article":  "Some text..."
    "summary":  "Summary of some text..."
}
```

## 0. Load imports

In [33]:
import os

## 1. Loading the data

In [34]:
from datasets import load_dataset

ROOT = os.path.join(os.getcwd(), "..")
DATA = os.path.join(ROOT, "data")
PROCESSED = os.path.join(DATA, "processed")
SUMMARY = os.path.join(PROCESSED, "summary")

CNN = os.path.join(SUMMARY, "CNN_DM")
NS = os.path.join(SUMMARY, "NS")

dataset = load_dataset("csv", data_files={
    "test": os.path.join(CNN, "test.csv"),
    "train": os.path.join(CNN, "train.csv"),
    "validation": os.path.join(CNN, "validation.csv"),
})

# temporaryily limit each split
seed=42
dataset["train"] = dataset["train"].shuffle(seed=seed).select(range(500))
dataset["validation"] = dataset["validation"].shuffle(seed=seed).select(range(500))
dataset["test"] = dataset["test"].shuffle(seed=seed).select(range(500))

## 2. Fine-tuning a Model

In [35]:
from transformers import BartTokenizer, BartForConditionalGeneration

base_model = "facebook/bart-base"   # ideally "facebook/bart-large"

tokenizer = BartTokenizer.from_pretrained(base_model)
model = BartForConditionalGeneration.from_pretrained(base_model)

Loading weights: 100%|██████████| 259/259 [00:00<00:00, 4313.72it/s]


In [36]:
def tokenize(example):
    # convert article to tokens
    inputs = tokenizer(
        example["article"],
        max_length=256,
        truncation=True,
        padding=False
    )

    # convert summaries into tokens
    labels = tokenizer(
        example["summary"],
        max_length=128,
        truncation=True,
        padding=False
    )

    inputs["labels"] = labels["input_ids"]
    return inputs

tokenized = dataset.map(
    tokenize,
    batched=True,
    remove_columns=["article", "summary"]   # remove raw text
)

tokenized.set_format("torch")

Map: 100%|██████████| 500/500 [00:00<00:00, 870.63 examples/s]


In [37]:
from transformers import DataCollatorForSeq2Seq

collator = DataCollatorForSeq2Seq(
    tokenizer,
    model=model,
    padding=True
)

In [ ]:
from transformers import Seq2SeqTrainingArguments

learning_rate = 2e-5
batch_size = 4
num_epochs = 1  # ideally ~3 but memory constraints
weight_decay = 0.01

args = Seq2SeqTrainingArguments(
    output_dir="./bart-finetuned-summarisation",

    learning_rate=learning_rate,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    num_train_epochs=num_epochs,
    weight_decay=weight_decay,

    eval_strategy="epoch",
    save_strategy="epoch",

    predict_with_generate=False,    # computationally expensive 
    load_best_model_at_end=True,
    metric_for_best_model="rouge2",

    logging_steps=10
)

## 3. Evaluation

In [39]:
import evaluate
import numpy as np

rouge = evaluate.load("rouge")

def metrics(eval_pred):
    pred, labels = eval_pred

    # convert predicted IDs back into text
    decoded_pred = tokenize.batch_decode(
        pred, skip_special_tokens=True
    )
    
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)   # replace with padding token
    decoded_labels = tokenize.batch_decode(
        labels, skip_special_tokens=True
    )

    res = rouge.compute(
        predictions=decoded_pred,
        references=decoded_labels,
        use_stemmer=True    # use the core form
    )

    return {k: round(v, 3) for k, v in res.items()}

## 4. Training

In [41]:
from transformers import Seq2SeqTrainer

trainer = Seq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    processing_class=tokenizer,
    data_collator=collator,
    compute_metrics=metrics
)

trainer.train()

Epoch,Training Loss,Validation Loss


RuntimeError: MPS backend out of memory (MPS allocated: 4.49 GiB, other allocations: 4.57 GiB, max allowed: 9.07 GiB). Tried to allocate 768.00 KiB on private pool. Use PYTORCH_MPS_HIGH_WATERMARK_RATIO=0.0 to disable upper limit for memory allocations (may cause system failure).

## 5. Inference

In [ ]:
# sample article-summary from CNN-DailyMail's test dataset (entry no. 2)
article = "A drunk teenage boy had to be rescued by security after jumping into a lions' enclosure at a zoo in western India. Rahul Kumar, 17, clambered over the enclosure fence at the Kamla Nehru Zoological Park in Ahmedabad, and began running towards the animals, shouting he would 'kill them'. Mr Kumar explained afterwards that he was drunk and 'thought I'd stand a good chance' against the predators. Next level drunk: Intoxicated Rahul Kumar, 17, climbed into the lions' enclosure at a zoo in Ahmedabad and began running towards the animals shouting 'Today I kill a lion!' Mr Kumar had been sitting near the enclosure when he suddenly made a dash for the lions, surprising zoo security. The intoxicated teenager ran towards the lions, shouting: 'Today I kill a lion or a lion kills me!' A zoo spokesman said: 'Guards had earlier spotted him close to the enclosure but had no idea he was planing to enter it. 'Fortunately, there are eight moats to cross before getting to where the lions usually are and he fell into the second one, allowing guards to catch up with him and take him out. 'We then handed him over to the police.' Brave fool: Fortunately, Mr Kumar  fell into a moat as he ran towards the lions and could be rescued by zoo security staff before reaching the animals (stock image) Kumar later explained: 'I don't really know why I did it. 'I was drunk and thought I'd stand a good chance.' A police spokesman said: 'He has been cautioned and will be sent for psychiatric evaluation. 'Fortunately for him, the lions were asleep and the zoo guards acted quickly enough to prevent a tragedy similar to that in Delhi.' Last year a 20-year-old man was mauled to death by a tiger in the Indian capital after climbing into its enclosure at the city zoo."
summmary = "Drunk teenage boy climbed into lion enclosure at zoo in west India. Rahul Kumar, 17, ran towards animals shouting 'Today I kill a lion!' Fortunately he fell into a moat before reaching lions and was rescued."

inputs = tokenizer(
    article,
    return_tensor="pt",     # pytorch
    max_length=256,
    truncation=True
)

# generate summary
summary_ids = model.generate(
    inputs["input_ids"],
    max_length=128,
    min_length=32,
    num_beams=3,
    early_stopping=True
)
print(f"[DEBUG] summary_ids: {summary_ids}")

# convert summary tokens into text
res = tokenizer.decode(
    summary_ids[0],
    skip_special_tokens=True
)
print(res)

## 6. Final Evaluation

In [ ]:
trainer.args.predict_with_generate = True
results = trainer.evaluate(tokenized["test"])
print(results)